# 09 · Publish & Audit

**Layer:** cross-cutting &nbsp;|&nbsp; **Stage:** publish &nbsp;|&nbsp; **Target:** `banking_gold.pipeline_audit_log`

The pipeline's final task, wired with `run_if: All Done` so it executes
whether the run succeeded cleanly or diverted through quarantine. Its job is
to leave behind a record of what this batch actually produced: row counts per
layer, the verdict from both quality gates, and a one-line status for the
whole run.

Because it is the terminal task in the DAG, this is also where the demo
narrative lands — everything upstream is visible in one table.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.utils import AnalysisException

spark = SparkSession.builder.appName("medallion-publish-audit").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
BRONZE_DB = "banking_bronze"
SILVER_DB = "banking_silver"
GOLD_DB   = "banking_gold"

MANIFEST = [
    ("bronze", f"{BRONZE_DB}.raw_accounts"),
    ("bronze", f"{BRONZE_DB}.raw_transactions"),
    ("bronze", f"{BRONZE_DB}.quarantine_records"),
    ("silver", f"{SILVER_DB}.dim_account"),
    ("silver", f"{SILVER_DB}.txn_clean"),
    ("silver", f"{SILVER_DB}.txn_rejects"),
    ("gold",   f"{GOLD_DB}.account_balance_daily"),
    ("gold",   f"{GOLD_DB}.fraud_signal_summary"),
]

In [ ]:
rows = []

for layer, table in MANIFEST:
    try:
        count = spark.table(table).count()
        status = "PRESENT"
    except AnalysisException:
        # Quarantine only exists when the bronze gate failed - a missing table
        # here is expected on a clean run, not an error.
        count, status = 0, "ABSENT"
    rows.append((BATCH_ID, RUN_DATE, layer, table, count, status))
    print(f"  {layer:<7} {table:<38} {count:>8}  {status}")

In [ ]:
def gate_verdict(table):
    """Collapse a layer's dq_results into a single PASS/FAIL verdict."""
    try:
        dq = spark.table(table)
    except AnalysisException:
        return "NOT_RUN", 0, 0
    total  = dq.count()
    failed = dq.filter(F.col("status") == "FAIL").count()
    return ("FAIL" if failed else "PASS"), total, failed

bronze_verdict, bronze_checks, bronze_failed = gate_verdict(f"{BRONZE_DB}.dq_results")
silver_verdict, silver_checks, silver_failed = gate_verdict(f"{SILVER_DB}.dq_results")

print(f"bronze gate : {bronze_verdict}  ({bronze_failed}/{bronze_checks} checks failed)")
print(f"silver gate : {silver_verdict}  ({silver_failed}/{silver_checks} checks failed)")

rows.append((BATCH_ID, RUN_DATE, "bronze", "GATE:bronze_quality", bronze_checks, bronze_verdict))
rows.append((BATCH_ID, RUN_DATE, "silver", "GATE:silver_quality", silver_checks, silver_verdict))

In [ ]:
audit = (
    spark.createDataFrame(
        rows,
        "batch_id string, run_date string, stage string, table_name string, "
        "row_count bigint, status string",
    )
    .withColumn("logged_at", F.current_timestamp())
)

(
    audit.write.mode("append").format("parquet")
    .saveAsTable(f"{GOLD_DB}.pipeline_audit_log")
)

print(f"appended {audit.count()} audit rows to {GOLD_DB}.pipeline_audit_log")

In [ ]:
RUN_STATUS = "SUCCESS" if bronze_verdict == "PASS" and silver_verdict == "PASS" else "DEGRADED"

gold_rows = sum(r[4] for r in rows if r[2] == "gold" and not r[3].startswith("GATE:"))

print("=" * 66)
print(f"  MEDALLION BANKING PIPELINE - {BATCH_ID}")
print("=" * 66)
print(f"  run date        : {RUN_DATE}")
print(f"  bronze gate     : {bronze_verdict}")
print(f"  silver gate     : {silver_verdict}")
print(f"  gold rows built : {gold_rows}")
print(f"  RUN_STATUS      : {RUN_STATUS}")
print("=" * 66)

In [ ]:
spark.table(f"{GOLD_DB}.pipeline_audit_log") \
     .filter(F.col("batch_id") == BATCH_ID) \
     .orderBy("stage", "table_name") \
     .show(50, truncate=False)